# Lasso feature selection

Load the dataset with the package pipeline, hold out a test set of activities, select the features with `ModelingLasso` on the training set, inspect the regularisation path with `model.plot` (`TrailPredictor.plot.lasso.PlotLasso`), then score the model on the test activities.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from sklearn import config_context
from sklearn.metrics import root_mean_squared_error

from TrailPredictor.dataset import Dataset
from TrailPredictor.modeling.base_model import GROUP_COLUMN
from TrailPredictor.modeling.lasso import LassoConfig

## Dataset

`Dataset.load` reads the FIT files (cached in `data/computed`) and rebuilds the features.

In [2]:
dataset = Dataset.load(min_distance=40, max_distance=100, segment_width=500)
dataset.target = "pace"  # set before building the model: the result is saved per target

config = LassoConfig(
    seed=42,
    n_splits=5,  # outer folds: RMSE of the selection
    inner_n_splits=5,  # inner folds of LassoCV: choice of the penalty
    test_size=0.2,
)

f"{len(dataset.data)} segments, {dataset.data[GROUP_COLUMN].nunique()} activities"

'2070 segments, 20 activities'

## Train / test split

Whole activities go to one side or the other, so no segment of a test activity is seen during the selection.

In [3]:
train, test = dataset.split_test_train(
    test_size=config.test_size, random_state=config.seed
)

pd.DataFrame(
    {
        "segments": [len(train.data), len(test.data)],
        "activities": [
            train.data[GROUP_COLUMN].nunique(),
            test.data[GROUP_COLUMN].nunique(),
        ],
    },
    index=["train", "test"],
)

,segments,activities
train,1572,16
test,498,4


## Lasso selection on the training set

Each outer `GroupKFold` fold fits the pipeline preprocess → standardisation → `SelectFromModel(LassoCV)` → `LinearRegression`, the penalty being chosen by inner folds also grouped by activity. The result is saved under `data/computed/models/lasso/<dataset hash>/<target>/<config hash>.csv` and reloaded for the same dataset and `LassoConfig`.

In [4]:
model = train.modeling.lasso(config)
selection = model.evaluate_selection() if model.result is None else model.result

selection.groupby("fold")[["alpha", "rmse"]].first()

,alpha,rmse
fold,,
0,0.026645,2.019929
1,0.053483,2.084093
2,0.011630,2.280210
3,0.012666,2.173641
4,0.012590,1.286894


In [5]:
cv_rmse = selection.groupby("fold")["rmse"].first()
f"RMSE CV (train) : {cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}"

'RMSE CV (train) : 1.9690 ± 0.3936'

In [6]:
# share of the outer folds selecting each feature
model.selection_frequency

,frequency,mean_abs_coef
feature,,
drop_pos,1.0,2.011864e+00
pause,1.0,1.678824e+00
heart_rate_zone_Récupération,1.0,7.974975e-01
heart_rate_zone_Aérobic (Endur.),1.0,4.802948e-01
altitude_drop_neg_front_10km,1.0,4.781319e-01
drop_neg,1.0,3.231103e-01
heart_rate_zone_Seuil,1.0,2.513685e-01
altitude_drop_neg,1.0,1.792315e-01
altitude_drop_neg_back_5km,1.0,1.152320e-01


## Regularisation path

`model.plot` returns the plotting interface of the model (`PlotLasso`). Its `path` fits the selection once on the whole training set (activity ids routed as `groups` to the inner folds of `LassoCV`) and recomputes the coefficients on the `LassoCV` grid of penalties; keep one plotter so the path is not refitted for every figure.

In [7]:
lasso_plot = model.plot  # one plotter: its regularisation path is fitted once

# features in colour: the active ones at the 1-SE α (n_highlight=... to change)
lasso_plot.regularization_path().show()

In [8]:
path = lasso_plot.path
active = path.coef_path.abs() > path.threshold
# α decreasing: the first α where a feature is active is its entry point
entry_alpha = active.idxmax()
entered = active.any()
n_active_cv = int((path.coefs_at(path.alpha).abs() > path.threshold).sum())
n_active_1se = int((path.coefs_at(path.alpha_1se).abs() > path.threshold).sum())
first_features = entry_alpha[entered].sort_values(ascending=False).index[:6]

print(
    "- Premières variables à entrer dans le modèle quand α diminue : "
    + ", ".join(f"`{feature}`" for feature in first_features)
    + ".\n"
    f"- À l'α retenu ({path.alpha:.4g}), {n_active_cv} variables sont "
    f"actives ; avec la règle « 1 erreur-type » (α = {path.alpha_1se:.4g}), "
    f"{n_active_1se} variables suffiraient pour une erreur CV "
    "équivalente à une erreur-type près."
)

- Premières variables à entrer dans le modèle quand α diminue : `drop_pos`, `pause`, `heart_rate_zone_Récupération`, `altitude_drop_neg_front_10km`, `heart_rate_zone_Seuil`, `heart_rate_zone_Aérobic (Endur.)`.
- À l'α retenu (0.02269), 24 variables sont actives ; avec la règle « 1 erreur-type » (α = 0.2116), 9 variables suffiraient pour une erreur CV équivalente à une erreur-type près.


## Test set

Relaxed Lasso refitted on the whole training set (activity ids routed as `groups` to the inner folds of `LassoCV`), scored on the held-out activities.

In [9]:
pipeline = model.build_pipeline()
with config_context(enable_metadata_routing=True):
    pipeline.fit(train.predictors, train.target, groups=train.data[GROUP_COLUMN])

test_pred = pd.Series(
    pipeline.predict(test.predictors), index=test.data.index, name=test.prediction_label
)
test.prediction = test_pred
test_rmse = root_mean_squared_error(test.target, test_pred)

selected = pipeline.named_steps["selection"].get_feature_names_out()
print(
    f"RMSE test : {test_rmse:.4f} (RMSE CV train : {cv_rmse.mean():.4f})\n"
    f"{len(selected)} variables retenues sur {len(path.coef_path.columns)}"
)

RMSE test : 1.4064 (RMSE CV train : 1.9690)
24 variables retenues sur 48


In [10]:
test.plot.vs_prediction().show()